# DenseNet169 — CheXpert Fine-tuning (from ChestX-ray14 checkpoint)

**Source model:** DenseNet169 pre-trained on ChestX-ray14 (14 classes)  
**Target dataset:** CheXpert v1.0 — 223,414 images, 14 pathology labels  
**Test set:** CheXlocalize (`test_labels.csv`)  
**Uncertainty policy:** `-1` (uncertain) → treated as `0` (negative)  
**Views:** Frontal only (PA + AP) — lateral excluded  
**Hardware target:** Single GPU ≤ 8 GB VRAM  

---
### Directory structure expected
```
CHEXPERT_ROOT/
├── CheXpert-v1.0/
│   ├── train.csv
│   ├── valid.csv
│   └── train/
│       └── patient00001/study1/view1_frontal.jpg  ...
│
CHEXLOCALIZE_ROOT/
├── test_labels.csv
└── <images referenced by Path column>

OUTPUT_DIR/
├── best_model.pth
├── test_metrics.csv
├── learning_curves.png
├── roc_curves.png
├── pr_curves.png
├── confusion_matrices.png
└── auc_bar.png
```

## 0. Imports & Reproducibility

In [ ]:
import os
import random
import shutil
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast

import torchvision.transforms as T
import torchvision.models as models

from PIL import Image
from sklearn.metrics import (
    roc_auc_score, f1_score, accuracy_score,
    recall_score, precision_score,
    roc_curve, precision_recall_curve,
    multilabel_confusion_matrix,
)
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')

# ── Reproducibility ────────────────────────────────────────────────────────────
SEED = 42

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)

# ── Device ─────────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'  GPU  : {torch.cuda.get_device_name(0)}')
    print(f'  VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 1. Configuration

In [ ]:
# ─── CHANGE THESE PATHS ───────────────────────────────────────────────────────
# Root that contains the CheXpert-v1.0/ subfolder and CSVs
CHEXPERT_ROOT     = Path(r'D:\CheXpert')              # Windows example
# CHEXPERT_ROOT   = Path('/data/CheXpert')            # Linux / macOS

# Root that contains test_labels.csv and its images
CHEXLOCALIZE_ROOT = Path(r'D:\CheXlocalize')          # Windows example
# CHEXLOCALIZE_ROOT = Path('/data/CheXlocalize')      # Linux / macOS

# ChestX-ray14 checkpoint to start from (produced by the previous notebook)
PRETRAINED_CKPT   = Path('./checkpoints/best_model.pth')

# All outputs go here
OUTPUT_DIR        = Path('./chexpert_output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# ──────────────────────────────────────────────────────────────────────────────

TRAIN_CSV = CHEXPERT_ROOT / 'CheXpert-v1.0' / 'train.csv'
VALID_CSV = CHEXPERT_ROOT / 'CheXpert-v1.0' / 'valid.csv'
TEST_CSV  = CHEXLOCALIZE_ROOT / 'test_labels.csv'

# ── CheXpert label columns (14 pathologies) ───────────────────────────────────
# NOTE: These exactly match the CSV column names.
# 'No Finding' is excluded from multi-label targets — it is the absence label.
LABELS: List[str] = [
    'Enlarged Cardiomediastinum',
    'Cardiomegaly',
    'Lung Opacity',
    'Lung Lesion',
    'Edema',
    'Consolidation',
    'Pneumonia',
    'Atelectasis',
    'Pneumothorax',
    'Effusion',
    'Pleural Other',
    'Fracture',
    'Support Devices',
    'No Finding',        # kept as a target label (valid for multi-label)
]
# CheXpert competition uses these 5 for leaderboard evaluation
COMPETITION_LABELS: List[str] = [
    'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Pleural Other'
]
NUM_CLASSES = len(LABELS)

# ── Uncertainty handling ──────────────────────────────────────────────────────
# -1 (uncertain) → 0  (U-zeroes policy — simplest, widely used baseline)
# Alternative: U-ones (-1 → 1) or U-ignore (mask uncertain from loss).
# U-zeroes is conservative and avoids false positive signal from uncertain labels.
UNCERTAINTY_POLICY = 'U-zeroes'   # options: 'U-zeroes' | 'U-ones'

# ── Training ──────────────────────────────────────────────────────────────────
IMG_SIZE     = 224
BATCH_SIZE   = 32       # reduce to 16 if OOM on 8 GB VRAM
NUM_WORKERS  = 4
PIN_MEMORY   = True

# CheXpert is ~7x larger than ChestX-ray14 — fewer epochs needed
# One epoch ≈ 5,900 batches (191k frontal images / 32); 5 epochs ≈ 29.5k steps
LR_BACKBONE  = 5e-5    # very low — features from ChestX-ray14 are already relevant
LR_HEAD      = 5e-4    # head adapts from 14-NIH-classes to 14-CheXpert-classes
WEIGHT_DECAY = 1e-5
NUM_EPOCHS   = 5       # 5 epochs on 191k images >> 30 epochs on 78k
PATIENCE     = 3       # early stopping
USE_AMP      = True    # mixed precision — critical for 8 GB VRAM
THRESHOLD    = 0.5

for p in [TRAIN_CSV, VALID_CSV]:
    assert p.exists(), f'Not found: {p}'

print('Configuration ready.')
print(f'  CheXpert root    : {CHEXPERT_ROOT}')
print(f'  CheXlocalize root: {CHEXLOCALIZE_ROOT}')
print(f'  Pretrained ckpt  : {PRETRAINED_CKPT}')
print(f'  Output dir       : {OUTPUT_DIR}')
print(f'  Classes          : {NUM_CLASSES}')
print(f'  Uncertainty      : {UNCERTAINTY_POLICY}')

## 2. Data Loading & Preprocessing

In [ ]:
def resolve_image_path(csv_path_value: str, root: Path) -> Path:
    """
    CheXpert CSV paths look like:
      'CheXpert-v1.0/train/patient00001/study1/view1_frontal.jpg'
    These are relative to CHEXPERT_ROOT.

    CheXlocalize CSV paths may be structured differently — we try
    both a direct join and stripping a known prefix.
    """
    p = root / csv_path_value
    if p.exists():
        return p
    # Try stripping leading 'CheXpert-v1.0/' if root already contains it
    stripped = Path(csv_path_value).relative_to('CheXpert-v1.0') if csv_path_value.startswith('CheXpert-v1.0') else Path(csv_path_value)
    p2 = root / stripped
    if p2.exists():
        return p2
    return root / csv_path_value   # return best guess; will fail at open() with clear error


def apply_uncertainty_policy(df: pd.DataFrame, policy: str) -> pd.DataFrame:
    """
    U-zeroes: -1 → 0  (treat uncertain as negative)
    U-ones  : -1 → 1  (treat uncertain as positive)
    NaN     : always → 0 (label not mentioned = absent)
    """
    df = df.copy()
    fill_val = 0.0 if policy == 'U-zeroes' else 1.0
    for lbl in LABELS:
        if lbl in df.columns:
            df[lbl] = df[lbl].fillna(0.0).replace(-1.0, fill_val).astype(np.float32)
        else:
            # CheXlocalize test set may not have all columns — default to 0
            df[lbl] = 0.0
    return df


def load_chexpert_split(
    csv_path:  Path,
    root:      Path,
    policy:    str = UNCERTAINTY_POLICY,
    frontal_only: bool = True,
) -> pd.DataFrame:
    df = pd.read_csv(csv_path)

    if frontal_only and 'Frontal/Lateral' in df.columns:
        before = len(df)
        df = df[df['Frontal/Lateral'] == 'Frontal'].reset_index(drop=True)
        print(f'  Frontal filter: {before:,} → {len(df):,} rows')

    df = apply_uncertainty_policy(df, policy)

    # Resolve absolute image paths
    df['abs_path'] = df['Path'].apply(lambda p: str(resolve_image_path(p, root)))
    print(f'  Loaded {len(df):,} rows from {csv_path.name}')
    return df


def load_chexlocalize_test(
    csv_path:  Path,
    root:      Path,
    policy:    str = UNCERTAINTY_POLICY,
) -> pd.DataFrame:
    """
    CheXlocalize test_labels.csv uses the same Path format.
    It contains only the 5 competition labels; missing ones default to 0.
    """
    df = pd.read_csv(csv_path)
    df = apply_uncertainty_policy(df, policy)
    df['abs_path'] = df['Path'].apply(lambda p: str(resolve_image_path(p, root)))
    print(f'  CheXlocalize test: {len(df):,} rows from {csv_path.name}')
    return df


print('Loading splits...')
train_df = load_chexpert_split(TRAIN_CSV, CHEXPERT_ROOT, frontal_only=True)
val_df   = load_chexpert_split(VALID_CSV, CHEXPERT_ROOT, frontal_only=True)
test_df  = load_chexlocalize_test(TEST_CSV, CHEXLOCALIZE_ROOT)

### 2.1 Label Distribution

In [ ]:
def plot_label_distribution(splits: Dict[str, pd.DataFrame]) -> None:
    n = len(splits)
    fig, axes = plt.subplots(1, n, figsize=(8 * n, 4), sharey=False)
    if n == 1:
        axes = [axes]
    for ax, (name, df_s) in zip(axes, splits.items()):
        available = [l for l in LABELS if l in df_s.columns]
        counts = df_s[available].sum().sort_values(ascending=False)
        sns.barplot(x=counts.index, y=counts.values, palette='Blues_r', ax=ax)
        ax.set_title(f'{name}  ({len(df_s):,} frontal images)', fontsize=11)
        ax.set_ylabel('Positive samples')
        ax.tick_params(axis='x', rotation=55)
        for bar, v in zip(ax.patches, counts.values):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + counts.max() * 0.012,
                f'{int(v):,}', ha='center', va='bottom', fontsize=6.5
            )
    plt.suptitle('Positive-label counts per split (CheXpert + CheXlocalize)', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'label_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()


plot_label_distribution({'Train': train_df, 'Val': val_df, 'Test (CheXlocalize)': test_df})

print('\nPositive rates (%) — Train vs Val vs Test:')
rate_data = {}
for split_name, df_s in [('Train %', train_df), ('Val %', val_df), ('Test %', test_df)]:
    rate_data[split_name] = (df_s[LABELS].mean() * 100).round(2)
display(pd.DataFrame(rate_data))

## 3. Dataset & DataLoaders

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]


def get_transforms(split: str) -> T.Compose:
    """
    CheXpert images are already 320x320 JPEGs.
    Training: random crop + flip + colour jitter for regularisation.
    Val/Test: deterministic centre crop.
    """
    if split == 'train':
        return T.Compose([
            T.Resize(256),
            T.RandomCrop(IMG_SIZE),
            T.RandomHorizontalFlip(p=0.5),
            T.RandomRotation(degrees=10),
            T.ColorJitter(brightness=0.15, contrast=0.15),
            T.ToTensor(),
            T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])
    return T.Compose([
        T.Resize((IMG_SIZE, IMG_SIZE)),
        T.ToTensor(),
        T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])


class CheXpertDataset(Dataset):
    """
    Memory-efficient: reads images on-the-fly from JPEG files.
    Label matrix (N x NUM_CLASSES float32) is the only RAM allocation.
    """

    def __init__(
        self,
        df:        pd.DataFrame,
        transform: Optional[T.Compose] = None,
    ) -> None:
        self.paths     = df['abs_path'].tolist()
        self.labels    = df[LABELS].values.astype(np.float32)   # (N, NUM_CLASSES)
        self.transform = transform

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        img = Image.open(self.paths[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        label = torch.from_numpy(self.labels[idx])   # (NUM_CLASSES,) float32
        return img, label


def build_loaders(
    train_df: pd.DataFrame,
    val_df:   pd.DataFrame,
    test_df:  pd.DataFrame,
) -> Dict[str, DataLoader]:
    splits = {'train': (train_df, 'train'), 'val': (val_df, 'val'), 'test': (test_df, 'test')}
    loaders: Dict[str, DataLoader] = {}
    for name, (df_s, split_key) in splits.items():
        ds = CheXpertDataset(df_s, get_transforms(split_key))
        bs = BATCH_SIZE if name == 'train' else BATCH_SIZE * 2
        loaders[name] = DataLoader(
            ds,
            batch_size         = bs,
            shuffle            = (name == 'train'),
            num_workers        = NUM_WORKERS,
            pin_memory         = PIN_MEMORY and DEVICE.type == 'cuda',
            drop_last          = (name == 'train'),
            persistent_workers = NUM_WORKERS > 0,
        )
        print(f'{name:5s} — {len(ds):>7,} samples | {len(loaders[name]):>5,} batches | bs={bs}')
    return loaders


loaders = build_loaders(train_df, val_df, test_df)

## 4. Model — DenseNet169 with ChestX-ray14 Weights

In [ ]:
def build_model(
    num_classes:       int  = NUM_CLASSES,
    pretrained_ckpt:   Optional[Path] = PRETRAINED_CKPT,
    chestxray14_classes: int = 14,
) -> nn.Module:
    """
    Builds DenseNet169 and loads weights from the ChestX-ray14 checkpoint.

    Transfer strategy:
      1. Load ImageNet DenseNet169 → replace head with 14-class ChestX-ray14 head.
      2. Load ChestX-ray14 state_dict (backbone + 14-class head).
      3. Replace the 14-class head with a fresh NUM_CLASSES head for CheXpert.

    This preserves all convolutional features learned on ChestX-ray14
    while initialising the new head randomly for the new label space.
    """
    # Step 1: ImageNet DenseNet169 with ChestX-ray14 head shape
    model = models.densenet169(weights=None)   # no ImageNet weights — we'll overwrite
    in_feat = model.classifier.in_features    # 1664
    model.classifier = nn.Linear(in_feat, chestxray14_classes)

    # Step 2: Load ChestX-ray14 weights (if checkpoint exists)
    if pretrained_ckpt is not None and pretrained_ckpt.exists():
        ckpt = torch.load(pretrained_ckpt, map_location='cpu')
        state_dict = ckpt.get('state_dict', ckpt)   # handle both raw and wrapped
        model.load_state_dict(state_dict, strict=True)
        epoch_info = ckpt.get('epoch', '?')
        auc_info   = ckpt.get('val_auc', '?')
        print(f'Loaded ChestX-ray14 checkpoint — epoch {epoch_info}, val AUC {auc_info}')
    else:
        print('WARNING: ChestX-ray14 checkpoint not found — falling back to ImageNet weights.')
        model = models.densenet169(weights=models.DenseNet169_Weights.IMAGENET1K_V1)
        model.classifier = nn.Linear(in_feat, chestxray14_classes)

    # Step 3: Replace head for CheXpert
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.5),                        # dropout regularises the large classifier
        nn.Linear(in_feat, num_classes),
        # No sigmoid — BCEWithLogitsLoss handles it
    )
    print(f'New classifier head: {model.classifier}')
    return model


model     = build_model().to(DEVICE)
total_p   = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTotal params     : {total_p:,}')
print(f'Trainable params : {trainable:,}')

## 5. Loss, Optimiser & Scheduler

In [ ]:
def compute_pos_weights(df: pd.DataFrame) -> torch.Tensor:
    """
    pos_weight_i = n_negative_i / n_positive_i.
    Clipped at 10 to prevent extreme weights for very rare labels
    from destabilising training on a dataset this large.
    """
    pos = df[LABELS].sum().values.astype(np.float64)
    neg = len(df) - pos
    w   = neg / np.clip(pos, 1, None)
    w   = np.clip(w, 1.0, 10.0)    # cap at 10× to keep loss scale stable
    return torch.tensor(w, dtype=torch.float32).to(DEVICE)


pos_weights = compute_pos_weights(train_df)
criterion   = nn.BCEWithLogitsLoss(pos_weight=pos_weights)

# Differential LRs — very conservative backbone LR because ChestX-ray14 features
# are already domain-relevant; we want fine adjustment, not catastrophic forgetting
backbone_params = [p for n, p in model.named_parameters() if 'classifier' not in n]
head_params     = list(model.classifier.parameters())

optimizer = optim.AdamW(
    [{'params': backbone_params, 'lr': LR_BACKBONE},
     {'params': head_params,     'lr': LR_HEAD}],
    weight_decay=WEIGHT_DECAY,
)

# ReduceLROnPlateau is better than cosine for large datasets:
# it responds to actual plateau signals rather than reducing on a fixed schedule
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=1, verbose=True
)

scaler = GradScaler(enabled=USE_AMP)

print('pos_weights (clipped at 10):')
for lbl, w in zip(LABELS, pos_weights.cpu().numpy()):
    print(f'  {lbl:<30s}: {w:.2f}')

## 6. Training Loop

In [ ]:
def train_one_epoch(
    model:     nn.Module,
    loader:    DataLoader,
    optimizer: optim.Optimizer,
    criterion: nn.Module,
    scaler:    GradScaler,
) -> float:
    model.train()
    total_loss = 0.0
    for imgs, labels in tqdm(loader, desc='Train', leave=False):
        imgs, labels = (
            imgs.to(DEVICE, non_blocking=True),
            labels.to(DEVICE, non_blocking=True),
        )
        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=USE_AMP):
            loss = criterion(model(imgs), labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(
    model:     nn.Module,
    loader:    DataLoader,
    criterion: nn.Module,
) -> Tuple[float, float]:
    """
    Returns (mean_loss, mean_auc_across_present_classes).
    Probs accumulated on CPU to avoid VRAM growth.
    """
    model.eval()
    total_loss      = 0.0
    all_probs: List[torch.Tensor]  = []
    all_labels: List[torch.Tensor] = []

    for imgs, labels in tqdm(loader, desc='Eval ', leave=False):
        imgs, labels = (
            imgs.to(DEVICE, non_blocking=True),
            labels.to(DEVICE, non_blocking=True),
        )
        with autocast(enabled=USE_AMP):
            logits = model(imgs)
            loss   = criterion(logits, labels)
        total_loss += loss.item()
        all_probs.append(logits.sigmoid().cpu())
        all_labels.append(labels.cpu())

    probs  = torch.cat(all_probs).numpy()    # (N, NUM_CLASSES)
    truths = torch.cat(all_labels).numpy()   # (N, NUM_CLASSES)

    aucs = [
        roc_auc_score(truths[:, i], probs[:, i])
        for i in range(NUM_CLASSES)
        if truths[:, i].sum() > 0 and truths[:, i].sum() < len(truths)
    ]
    return total_loss / len(loader), float(np.mean(aucs)) if aucs else 0.0


def fit(
    model:      nn.Module,
    loaders:    Dict[str, DataLoader],
    optimizer:  optim.Optimizer,
    scheduler,
    criterion:  nn.Module,
    scaler:     GradScaler,
    num_epochs: int  = NUM_EPOCHS,
    patience:   int  = PATIENCE,
    ckpt_path:  Path = OUTPUT_DIR / 'best_model.pth',
) -> Dict[str, List[float]]:

    history: Dict[str, List[float]] = {'train_loss': [], 'val_loss': [], 'val_auc': []}
    best_auc, stale = -1.0, 0

    for epoch in range(1, num_epochs + 1):
        print(f'\n── Epoch {epoch}/{num_epochs} ──────────────────────────')
        tr_loss           = train_one_epoch(model, loaders['train'], optimizer, criterion, scaler)
        val_loss, val_auc = evaluate(model, loaders['val'], criterion)
        scheduler.step(val_auc)   # ReduceLROnPlateau monitors val_auc

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(val_loss)
        history['val_auc'].append(val_auc)

        improved = val_auc > best_auc
        tag      = ' ← best' if improved else ''
        print(f'train_loss={tr_loss:.4f}  val_loss={val_loss:.4f}  val_auc={val_auc:.4f}{tag}')

        if improved:
            best_auc = val_auc
            stale    = 0
            torch.save({
                'epoch'      : epoch,
                'state_dict' : model.state_dict(),
                'optimizer'  : optimizer.state_dict(),
                'val_auc'    : best_auc,
                'labels'     : LABELS,
                'num_classes': NUM_CLASSES,
            }, ckpt_path)
            print(f'  Checkpoint saved → {ckpt_path}')
        else:
            stale += 1
            if stale >= patience:
                print(f'Early stopping triggered at epoch {epoch}.')
                break

    print(f'\nBest val AUC : {best_auc:.4f}')
    return history


history = fit(model, loaders, optimizer, scheduler, criterion, scaler)

## 7. Learning Curves

In [ ]:
def plot_learning_curves(history: Dict[str, List[float]]) -> None:
    epochs = range(1, len(history['train_loss']) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(epochs, history['train_loss'], marker='o', ms=5, label='Train')
    ax1.plot(epochs, history['val_loss'],   marker='o', ms=5, label='Val')
    ax1.set(title='Loss', xlabel='Epoch', ylabel='BCE Loss')
    ax1.legend(); ax1.grid(alpha=0.3)
    ax1.set_xticks(list(epochs))

    best = max(history['val_auc'])
    ax2.plot(epochs, history['val_auc'], marker='o', ms=5, color='darkorange', label='Val AUC')
    ax2.axhline(best, ls='--', color='red', alpha=0.7, label=f'Best = {best:.4f}')
    ax2.set(title='Validation Mean AUC', xlabel='Epoch', ylabel='AUC-ROC')
    ax2.legend(); ax2.grid(alpha=0.3)
    ax2.set_xticks(list(epochs))

    plt.suptitle('DenseNet169 — CheXpert Fine-tuning (from ChestX-ray14)', fontsize=13)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'learning_curves.png', dpi=150)
    plt.show()
    print(f'Saved → {OUTPUT_DIR / "learning_curves.png"}')

plot_learning_curves(history)

## 8. Load Best Checkpoint & Run CheXlocalize Test Inference

In [ ]:
ckpt = torch.load(OUTPUT_DIR / 'best_model.pth', map_location=DEVICE)
model.load_state_dict(ckpt['state_dict'])
print(f'Loaded epoch {ckpt["epoch"]} — val AUC {ckpt["val_auc"]:.4f}')


@torch.no_grad()
def predict(
    model:  nn.Module,
    loader: DataLoader,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Returns (y_prob, y_true) — shape (N, NUM_CLASSES).
    Probabilities accumulated on CPU to avoid VRAM pressure.
    """
    model.eval()
    all_probs:  List[torch.Tensor] = []
    all_labels: List[torch.Tensor] = []
    for imgs, labels in tqdm(loader, desc='Test inference'):
        imgs = imgs.to(DEVICE, non_blocking=True)
        with autocast(enabled=USE_AMP):
            logits = model(imgs)
        all_probs.append(logits.sigmoid().cpu())
        all_labels.append(labels)
    return (
        torch.cat(all_probs).numpy(),
        torch.cat(all_labels).numpy(),
    )


y_prob, y_true = predict(model, loaders['test'])
y_pred = (y_prob >= THRESHOLD).astype(int)
print(f'Test predictions — shape: {y_prob.shape}')

## 9. Per-Label Metrics

In [ ]:
def compute_metrics(
    y_true:        np.ndarray,
    y_prob:        np.ndarray,
    y_pred:        np.ndarray,
    competition_labels: List[str] = COMPETITION_LABELS,
) -> pd.DataFrame:
    rows = []
    for i, lbl in enumerate(LABELS):
        gt, pb, pd_ = y_true[:, i], y_prob[:, i], y_pred[:, i]
        n_pos = int(gt.sum())
        # Skip AUC if class is constant (all 0 or all 1)
        auc = (
            round(roc_auc_score(gt, pb), 4)
            if 0 < n_pos < len(gt) else float('nan')
        )
        rows.append({
            'Label'       : lbl,
            'Competition' : '★' if lbl in competition_labels else '',
            'N_pos'       : n_pos,
            'Prevalence'  : f'{gt.mean()*100:.2f}%',
            'AUC'         : auc,
            'F1'          : round(f1_score(gt, pd_, zero_division=0),        4),
            'Accuracy'    : round(accuracy_score(gt, pd_),                   4),
            'Recall'      : round(recall_score(gt, pd_, zero_division=0),    4),
            'Precision'   : round(precision_score(gt, pd_, zero_division=0), 4),
        })

    mdf = pd.DataFrame(rows)

    # Macro average (all labels)
    macro = {
        'Label': 'MACRO AVG', 'Competition': '',
        'N_pos': int(y_true.sum()), 'Prevalence': f'{y_true.mean()*100:.2f}%',
        'AUC'      : round(mdf['AUC'].dropna().mean(), 4),
        'F1'       : round(mdf['F1'].mean(), 4),
        'Accuracy' : round(mdf['Accuracy'].mean(), 4),
        'Recall'   : round(mdf['Recall'].mean(), 4),
        'Precision': round(mdf['Precision'].mean(), 4),
    }
    # Competition-5 average
    comp_mask = mdf['Label'].isin(competition_labels)
    comp = {
        'Label': 'COMPETITION AVG (5)', 'Competition': '★',
        'N_pos': int(y_true[:, [LABELS.index(l) for l in competition_labels]].sum()),
        'Prevalence': '',
        'AUC'      : round(mdf.loc[comp_mask, 'AUC'].dropna().mean(), 4),
        'F1'       : round(mdf.loc[comp_mask, 'F1'].mean(), 4),
        'Accuracy' : round(mdf.loc[comp_mask, 'Accuracy'].mean(), 4),
        'Recall'   : round(mdf.loc[comp_mask, 'Recall'].mean(), 4),
        'Precision': round(mdf.loc[comp_mask, 'Precision'].mean(), 4),
    }
    return pd.concat([mdf, pd.DataFrame([macro, comp])], ignore_index=True)


metrics_df = compute_metrics(y_true, y_prob, y_pred)

num_cols = ['AUC', 'F1', 'Accuracy', 'Recall', 'Precision']
label_rows = metrics_df[~metrics_df['Label'].str.startswith('MACRO') &
                        ~metrics_df['Label'].str.startswith('COMPETITION')]

display(
    metrics_df.style
        .format({c: lambda x: f'{x:.4f}' if pd.notna(x) and isinstance(x, float) else str(x)
                 for c in num_cols})
        .highlight_max(subset=num_cols, color='#c6efce')
        .highlight_min(subset=num_cols, color='#ffeb9c')
        .set_caption('Per-Label Test Metrics — DenseNet169 / CheXpert → CheXlocalize')
)

metrics_df.to_csv(OUTPUT_DIR / 'test_metrics.csv', index=False)
print(f'Saved → {OUTPUT_DIR / "test_metrics.csv"}')

## 10. ROC Curves

In [ ]:
def plot_roc_curves(y_true: np.ndarray, y_prob: np.ndarray) -> None:
    # Only plot labels that have both positives and negatives
    plottable = [
        (i, lbl) for i, lbl in enumerate(LABELS)
        if 0 < y_true[:, i].sum() < len(y_true)
    ]
    n_cols = 4
    n_rows = (len(plottable) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 4.2))
    axes = axes.flatten()

    for ax_idx, (i, lbl) in enumerate(plottable):
        fpr, tpr, _ = roc_curve(y_true[:, i], y_prob[:, i])
        auc_val     = roc_auc_score(y_true[:, i], y_prob[:, i])
        is_comp     = lbl in COMPETITION_LABELS
        color       = 'darkorange' if is_comp else 'steelblue'
        ax = axes[ax_idx]
        ax.plot(fpr, tpr, lw=2, color=color,
                label=f'AUC = {auc_val:.3f}' + (' ★' if is_comp else ''))
        ax.fill_between(fpr, tpr, alpha=0.08, color=color)
        ax.plot([0, 1], [0, 1], 'k--', lw=1)
        ax.set(title=lbl, xlabel='FPR', ylabel='TPR', xlim=[0, 1], ylim=[0, 1.02])
        ax.legend(loc='lower right', fontsize=9)
        ax.grid(alpha=0.3)

    for j in range(len(plottable), len(axes)):
        axes[j].set_visible(False)

    plt.suptitle('ROC Curves — Per Pathology (CheXlocalize Test Set)\n★ = CheXpert competition labels',
                 fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'roc_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved → {OUTPUT_DIR / "roc_curves.png"}')

plot_roc_curves(y_true, y_prob)

## 11. Precision-Recall Curves

In [ ]:
def plot_pr_curves(y_true: np.ndarray, y_prob: np.ndarray) -> None:
    plottable = [
        (i, lbl) for i, lbl in enumerate(LABELS)
        if 0 < y_true[:, i].sum() < len(y_true)
    ]
    f1_lookup = dict(zip(
        metrics_df['Label'], metrics_df['F1']
    ))
    n_cols = 4
    n_rows = (len(plottable) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 4.2))
    axes = axes.flatten()

    for ax_idx, (i, lbl) in enumerate(plottable):
        prec, rec, _ = precision_recall_curve(y_true[:, i], y_prob[:, i])
        baseline     = y_true[:, i].mean()
        is_comp      = lbl in COMPETITION_LABELS
        color        = 'darkorange' if is_comp else '#5DCAA5'
        ax = axes[ax_idx]
        ax.plot(rec, prec, lw=2, color=color,
                label=f'F1={f1_lookup.get(lbl, 0):.3f}' + (' ★' if is_comp else ''))
        ax.fill_between(rec, prec, alpha=0.08, color=color)
        ax.axhline(baseline, ls='--', color='grey', lw=1,
                   label=f'Prevalence={baseline:.3f}')
        ax.set(title=lbl, xlabel='Recall', ylabel='Precision', xlim=[0, 1], ylim=[0, 1.05])
        ax.legend(loc='upper right', fontsize=9)
        ax.grid(alpha=0.3)

    for j in range(len(plottable), len(axes)):
        axes[j].set_visible(False)

    plt.suptitle('Precision-Recall Curves — Per Pathology (CheXlocalize Test Set)\n★ = CheXpert competition labels',
                 fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'pr_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved → {OUTPUT_DIR / "pr_curves.png"}')

plot_pr_curves(y_true, y_prob)

## 12. Confusion Matrices

In [ ]:
def plot_confusion_matrices(y_true: np.ndarray, y_pred: np.ndarray) -> None:
    # Only plot labels present in test set
    plottable_idx  = [i for i, lbl in enumerate(LABELS)
                      if 0 < y_true[:, i].sum() < len(y_true)]
    plottable_lbls = [LABELS[i] for i in plottable_idx]

    mcm    = multilabel_confusion_matrix(y_true, y_pred)   # (NUM_CLASSES, 2, 2)
    n_cols = 4
    n_rows = (len(plottable_idx) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 4))
    axes = axes.flatten()

    for ax_idx, (i, lbl) in enumerate(zip(plottable_idx, plottable_lbls)):
        cm      = mcm[i]
        cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(1)
        ax = axes[ax_idx]
        sns.heatmap(
            cm_norm, annot=cm, fmt='d', cmap='Blues',
            vmin=0, vmax=1, cbar=False, ax=ax,
            xticklabels=['Pred Neg', 'Pred Pos'],
            yticklabels=['True Neg', 'True Pos'],
            annot_kws={'size': 11},
        )
        title = lbl + (' ★' if lbl in COMPETITION_LABELS else '')
        ax.set_title(title, fontsize=10)

    for j in range(len(plottable_idx), len(axes)):
        axes[j].set_visible(False)

    plt.suptitle(
        f'Confusion Matrices  (threshold={THRESHOLD}) — CheXlocalize Test Set\n★ = CheXpert competition labels',
        fontsize=13, y=1.01
    )
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'confusion_matrices.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved → {OUTPUT_DIR / "confusion_matrices.png"}')

plot_confusion_matrices(y_true, y_pred)

## 13. AUC Summary Bar Chart

In [ ]:
def plot_auc_bar(metrics_df: pd.DataFrame) -> None:
    summary_labels = ['MACRO AVG', 'COMPETITION AVG (5)']
    df_p = (
        metrics_df[~metrics_df['Label'].isin(summary_labels)]
        .dropna(subset=['AUC'])
        .sort_values('AUC', ascending=False)
        .copy()
    )
    macro_auc = float(metrics_df.loc[metrics_df['Label'] == 'MACRO AVG', 'AUC'].iloc[0])
    comp_auc  = float(metrics_df.loc[metrics_df['Label'] == 'COMPETITION AVG (5)', 'AUC'].iloc[0])

    fig, ax = plt.subplots(figsize=(15, 5))
    bar_colors = [
        '#D85A30' if lbl in COMPETITION_LABELS else '#378ADD'
        for lbl in df_p['Label']
    ]
    bars = ax.bar(df_p['Label'], df_p['AUC'], color=bar_colors, edgecolor='black', lw=0.5)

    ax.axhline(macro_auc, ls='--', lw=2, color='navy',  label=f'Macro AUC = {macro_auc:.4f}')
    ax.axhline(comp_auc,  ls=':',  lw=2, color='#D85A30', label=f'Competition-5 AUC = {comp_auc:.4f}')
    ax.axhline(0.5,       ls=':',  lw=1, color='gray',  label='Random baseline = 0.50')

    for bar, val in zip(bars, df_p['AUC']):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.004,
            f'{val:.3f}', ha='center', va='bottom', fontsize=8
        )

    # Legend patches
    from matplotlib.patches import Patch
    legend_patches = [
        Patch(color='#D85A30', label='Competition label ★'),
        Patch(color='#378ADD', label='Other label'),
    ]
    legend1 = ax.legend(handles=legend_patches, loc='lower left', fontsize=9)
    ax.add_artist(legend1)
    ax.legend(loc='lower right', fontsize=9)

    ax.set(ylim=[0.4, 1.05], ylabel='AUC-ROC',
           title='Per-Pathology AUC-ROC — DenseNet169 / CheXpert → CheXlocalize Test Set')
    ax.tick_params(axis='x', rotation=45)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'auc_bar.png', dpi=150)
    plt.show()
    print(f'Saved → {OUTPUT_DIR / "auc_bar.png"}')

plot_auc_bar(metrics_df)

## 14. Final Summary

In [ ]:
macro = metrics_df[metrics_df['Label'] == 'MACRO AVG'].iloc[0]
comp  = metrics_df[metrics_df['Label'] == 'COMPETITION AVG (5)'].iloc[0]

print('=' * 66)
print('  DenseNet169 (ChestX-ray14 → CheXpert) — CheXlocalize Test Set')
print('=' * 66)
print('  All-14 Macro:')
print(f'    AUC       : {macro["AUC"]:.4f}')
print(f'    F1        : {macro["F1"]:.4f}')
print(f'    Accuracy  : {macro["Accuracy"]:.4f}')
print(f'    Recall    : {macro["Recall"]:.4f}')
print(f'    Precision : {macro["Precision"]:.4f}')
print()
print('  Competition-5 Macro (★):')
print(f'    AUC       : {comp["AUC"]:.4f}')
print(f'    F1        : {comp["F1"]:.4f}')
print('=' * 66)
print('\nPer-label AUC (sorted high → low):')
df_sorted = (
    metrics_df[
        ~metrics_df['Label'].isin(['MACRO AVG', 'COMPETITION AVG (5)'])
    ]
    .dropna(subset=['AUC'])
    .sort_values('AUC', ascending=False)
)
for _, row in df_sorted.iterrows():
    star = ' ★' if row['Label'] in COMPETITION_LABELS else '  '
    bar  = '█' * int(round(float(row['AUC']) * 30))
    print(f'  {row["Label"]:<30s}{star}  {row["AUC"]:.4f}  {bar}')
print('=' * 66)
print(f'\nAll outputs saved to: {OUTPUT_DIR.resolve()}')